<a href="https://colab.research.google.com/github/rubenchov/Transformers/blob/main/3scenesViT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports and drive

In [1]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from transformers import ViTImageProcessor, ViTForImageClassification, TrainingArguments, Trainer

#Drive
from google.colab import drive
drive.mount('/content/drive')
base_path = '/content/drive/MyDrive/2025/Docencia/Visión con IA/2. Aprendizaje Automático/3scenes/'
subfolders = ['coast', 'forest', 'highway']

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Import dataset and split

In [2]:
data, labels = [], []
for i, subfolder in enumerate(subfolders):
    path = os.path.join(base_path, subfolder)
    for file in filter(lambda f: f.endswith('.jpg'), os.listdir(path)):
        img = cv2.imread(os.path.join(path, file))
        img = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), (224, 224))
        data.append(img)
        labels.append(i)

#Split dataset
trainX, testX, trainY, testY = train_test_split(data, labels, test_size=0.25, random_state=40)



#Model definition

In [3]:
#Model initialization and processor
model_name = "google/vit-base-patch16-224"
processor = ViTImageProcessor.from_pretrained(model_name)
model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=len(subfolders),
    id2label={i: c for i, c in enumerate(subfolders)},
    label2id={c: i for i, c in enumerate(subfolders)},
    ignore_mismatched_sizes=True
)

#Adapt dataset
class CompactDataset(Dataset):
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        pixel_values = processor(images=self.images[idx], return_tensors="pt")["pixel_values"].squeeze(0)
        return {"pixel_values": pixel_values, "labels": torch.tensor(self.labels[idx], dtype=torch.long)}

train_ds = CompactDataset(trainX, trainY)
test_ds = CompactDataset(testX, testY)



[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([3, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([3])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


#Training

In [4]:
args = TrainingArguments(
    output_dir="./results_opt",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    remove_unused_columns=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=lambda p: {"accuracy": (np.argmax(p.predictions, axis=1) == p.label_ids).mean()}
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.038522,0.995833
2,No log,0.038926,0.995833
3,No log,0.039415,0.995833


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['vit.layers.0.attention.q_proj.weight', 'vit.layers.0.attention.q_proj.bias', 'vit.layers.0.attention.k_proj.weight', 'vit.layers.0.attention.k_proj.bias', 'vit.layers.0.attention.v_proj.weight', 'vit.layers.0.attention.v_proj.bias', 'vit.layers.0.attention.o_proj.weight', 'vit.layers.0.attention.o_proj.bias', 'vit.layers.0.layernorm_before.weight', 'vit.layers.0.layernorm_before.bias', 'vit.layers.0.layernorm_after.weight', 'vit.layers.0.layernorm_after.bias', 'vit.layers.0.mlp.fc1.weight', 'vit.layers.0.mlp.fc1.bias', 'vit.layers.0.mlp.fc2.weight', 'vit.layers.0.mlp.fc2.bias', 'vit.layers.1.attention.q_proj.weight', 'vit.layers.1.attention.q_proj.bias', 'vit.layers.1.attention.k_proj.weight', 'vit.layers.1.attention.k_proj.bias', 'vit.layers.1.attention.v_proj.weight', 'vit.layers.1.attention.v_proj.bias', 'vit.layers.1.attention.o_proj.weight', 'vit.layers.1.attention.o_proj.bias', 'vit.layers.1.layernorm_before

TrainOutput(global_step=270, training_loss=0.03678504449349863, metrics={'train_runtime': 11.5914, 'train_samples_per_second': 185.827, 'train_steps_per_second': 23.293, 'total_flos': 1.6691924166078874e+17, 'train_loss': 0.03678504449349863, 'epoch': 3.0})

#Validation

In [5]:
eval_res = trainer.evaluate()
print("\nResultados finales de evaluación:", eval_res)

Training Loss,Validation Loss,Epoch,Accuracy
No log,0.039302,3,0.995833



Resultados finales de evaluación: {'eval_loss': 0.03930177167057991, 'eval_accuracy': 0.9958333333333333}
